In [1]:
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio timm -q

In [2]:
!pip install sympy==1.12 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 55.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.


In [7]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class CUBDataset(Dataset):
    """
    Custom Dataset cho CUB-200-2011 lấy cả Ảnh, Label và Caption.
    Đã tối ưu cho trường hợp 1 ảnh = 1 file caption duy nhất.
    """
    def __init__(self, root_dir, text_dir=None, is_train=True, transform=None):
        self.root_dir = root_dir
        self.is_train = is_train
        self.transform = transform
        self.image_dir = os.path.join(root_dir, 'images')

        if text_dir is None:
            base_dataset_dir = os.path.dirname(root_dir) # /content/dataset_cub/
            self.text_dir = os.path.join(base_dataset_dir, 'captions')
        else:
            self.text_dir = text_dir

        images_txt = os.path.join(root_dir, 'images.txt')
        split_txt = os.path.join(root_dir, 'train_test_split.txt')
        labels_txt = os.path.join(root_dir, 'image_class_labels.txt')
        classes_txt = os.path.join(root_dir, 'classes.txt')

        images_df = pd.read_csv(images_txt, sep=' ', names=['img_id', 'filepath'])
        split_df = pd.read_csv(split_txt, sep=' ', names=['img_id', 'is_train'])
        labels_df = pd.read_csv(labels_txt, sep=' ', names=['img_id', 'label'])
        classes_df = pd.read_csv(classes_txt, sep=' ', names=['class_id', 'class_name'])

        df = images_df.merge(split_df, on='img_id').merge(labels_df, on='img_id')

        target_split = 1 if self.is_train else 0
        df = df[df['is_train'] == target_split]

        self.image_paths = df['filepath'].tolist()
        self.labels = (df['label'] - 1).tolist()
        self.class_map = {}
        for _, row in classes_df.iterrows():
            class_name = row.class_name
            class_name = class_name.split('.', 1)[1]
            class_name = class_name.replace('_', ' ').lower()

            self.class_map[row.class_id - 1] = class_name

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        rel_path = self.image_paths[idx]

        # image
        img_path = os.path.join(self.image_dir, rel_path)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # label
        label = self.labels[idx]

        # class name
        class_name = self.class_map[label]

        # caption
        text_rel_path = rel_path.replace('.jpg', '.txt')
        text_path = os.path.join(self.text_dir, text_rel_path)

        caption = f"a picture of {class_name}"
        if os.path.exists(text_path):
            with open(text_path, 'r', encoding='utf-8') as f:
                raw_caption = f.read().strip()

            caption = f"a picture of {class_name}. {raw_caption}"

        return image, label, caption.lower()

In [4]:
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
from PIL import Image

transform_resnet = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

transform_clip = transforms.Compose([
    transforms.Resize(224, interpolation=Image.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

In [10]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import timm
from transformers import AutoProcessor, AutoModel, AutoModelForImageClassification
from peft import PeftModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang tải các mô hình lên GPU...")


convnext_model = timm.create_model('convnextv2_tiny', pretrained=False, num_classes=200)
raw_state_dict = torch.load("/content/drive/MyDrive/ImageRetrieval/model/convnextv2/best_convnext_cub.pth", map_location=device)

# 1.2 Duyệt qua các key và xóa tiền tố "model." bị thừa
cleaned_state_dict = {}
for key, value in raw_state_dict.items():
    # Cắt bỏ 'model.' ở đầu nếu có
    new_key = key.replace("model.", "", 1) if key.startswith("model.") else key
    cleaned_state_dict[new_key] = value

# 1.3 Nạp state_dict đã làm sạch vào mô hình
convnext_model.load_state_dict(cleaned_state_dict)

convnext_model.reset_classifier(0)
convnext_model.to(device).eval()

# Transform chuẩn của ConvNeXtV2 (ImageNet)
transform_convnext = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dino_processor = AutoProcessor.from_pretrained("facebook/dinov2-base")
dino_base = AutoModel.from_pretrained("facebook/dinov2-base")

dino_model = PeftModel.from_pretrained(dino_base, "/content/drive/MyDrive/ImageRetrieval/model/dinov2/")
dino_model.to(device).eval()

SIGLIP_MODEL_NAME = "google/siglip-base-patch16-224"
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL_NAME)
siglip_base = AutoModel.from_pretrained(SIGLIP_MODEL_NAME)

siglip_model = PeftModel.from_pretrained(siglip_base, "/content/drive/MyDrive/ImageRetrieval/model/siglip/")
siglip_model.to(device).eval()

print("Đã tải xong ConvNeXtV2, DINOv2 và SigLIP2!")

Đang tải các mô hình lên GPU...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Đã tải xong ConvNeXtV2, DINOv2 và SigLIP2!


In [13]:
import os
from google.colab import drive
import os
import zipfile
import os
from tqdm import tqdm
db_path = "/content/drive/MyDrive/ImageRetrieval/database/cub_vector_database.pt"

if not os.path.exists(db_path):
    print("Database chưa tạo, bắt đầu tạo...")

    drive.mount('/content/drive')

    zip_path = '/content/drive/MyDrive/ImageRetrieval/dataset/CUB.zip'
    extract_path = '/content/dataset_cub'

    if not os.path.exists(extract_path):
        os.makedirs(extract_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            print("Đang giải nén... vui lòng đợi...")
            zip_ref.extractall(extract_path)
            print(f"Hoàn thành! Dữ liệu đã nằm tại: {extract_path}")
    else:
        print("Thư mục đã tồn tại, bỏ qua bước giải nén.")


    dataset_raw = CUBDataset(root_dir='/content/dataset_cub/CUB_200_2011', is_train=False, transform=None)
    loader_raw = DataLoader(dataset_raw, batch_size=32, shuffle=False, num_workers=2, collate_fn=lambda x: x)

    db_convnext, db_dino, db_combined, db_siglip = [], [], [], []
    db_image_paths = [item[0] for item in dataset_raw.image_paths] # Tạm lưu đường dẫn

    print("🚀 ĐANG XÂY DỰNG KHO DỮ LIỆU VECTOR (4 MÔ HÌNH)...")

    with torch.no_grad():
        for batch in tqdm(loader_raw, desc="Extracting Features"):
            images_pil = [item[0] for item in batch]

            # Trích xuất ConvNeXtV2
            inputs_conv = torch.stack([transform_convnext(img) for img in images_pil]).to(device)
            feats_conv = convnext_model(inputs_conv) # [Batch, 768]
            feats_conv_norm = feats_conv / feats_conv.norm(p=2, dim=-1, keepdim=True)
            db_convnext.append(feats_conv_norm.cpu())

            # 2Trích xuất DINOv2
            inputs_dino = dino_processor(images=images_pil, return_tensors="pt").to(device)
            outputs_dino = dino_model(**inputs_dino)
            # DINOv2 lấy [CLS] token ở vị trí 0 của last_hidden_state
            feats_dino = outputs_dino.last_hidden_state[:, 0, :] # [Batch, 768]
            feats_dino_norm = feats_dino / feats_dino.norm(p=2, dim=-1, keepdim=True)
            db_dino.append(feats_dino_norm.cpu())

            # ConvNeXt + DINOv2
            # Concat 2 vector đã chuẩn hóa dọc theo chiều feature (dim=-1)
            feats_combined = torch.cat([feats_conv_norm, feats_dino_norm], dim=-1) # [Batch, 1536]
            # Chuẩn hóa L2 lại lần nữa cho vector tổng hợp
            feats_combined_norm = feats_combined / feats_combined.norm(p=2, dim=-1, keepdim=True)
            db_combined.append(feats_combined_norm.cpu())

            # SigLIP2
            inputs_siglip = siglip_processor(images=images_pil, return_tensors="pt").to(device)
            feats_siglip = siglip_model.get_image_features(**inputs_siglip) # SigLIP có sẵn hàm này

            if hasattr(feats_siglip, 'pooler_output'):
                feats_siglip = feats_siglip.pooler_output

            feats_siglip_norm = feats_siglip / feats_siglip.norm(p=2, dim=-1, keepdim=True)
            db_siglip.append(feats_siglip_norm.cpu())

    # Nối batch
    tensor_db_convnext = torch.cat(db_convnext, dim=0)
    tensor_db_dino = torch.cat(db_dino, dim=0)
    tensor_db_combined = torch.cat(db_combined, dim=0)
    tensor_db_siglip = torch.cat(db_siglip, dim=0)

    db_image_paths = dataset_raw.image_paths

    # Lưu Database
    save_dir = "/content/drive/MyDrive/ImageRetrieval/database"
    os.makedirs(save_dir, exist_ok=True)

    torch.save({
        'convnext': tensor_db_convnext,
        'dinov2': tensor_db_dino,
        'combined': tensor_db_combined,
        'siglip': tensor_db_siglip,
        'paths': db_image_paths
    }, os.path.join(save_dir, "cub_vector_database.pt"))

    print(f"Hoàn thành! Đã lưu Database {len(db_image_paths)} ảnh (4 Models) vào Drive.")

Database chưa tạo, bắt đầu tạo...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục đã tồn tại, bỏ qua bước giải nén.
🚀 ĐANG XÂY DỰNG KHO DỮ LIỆU VECTOR (4 MÔ HÌNH)...


Extracting Features: 100%|██████████| 182/182 [04:56<00:00,  1.63s/it]


Hoàn thành! Đã lưu Database 5794 ảnh (4 Models) vào Drive.


# server


In [15]:
import os
import io
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from transformers import ViTForImageClassification, CLIPModel, CLIPProcessor
from peft import PeftModel

from fastapi import FastAPI, UploadFile, File, Form
import uvicorn
from pyngrok import ngrok
import nest_asyncio

# vector db
db_data = torch.load("/content/drive/MyDrive/ImageRetrieval/database/cub_vector_database.pt", map_location=device)

DB_CONVNEXT = db_data['convnext'].to(device)
DB_DINO     = db_data['dinov2'].to(device)
DB_COMBINED = db_data['combined'].to(device)
DB_SIGLIP   = db_data['siglip'].to(device)
IMAGE_PATHS = db_data['paths']
def retrieve_top_k(query_vector, db_matrix, top_k):
    # Tính Cosine Similarity
    similarities = torch.matmul(query_vector, db_matrix.T)
    values, indices = torch.topk(similarities.squeeze(0), top_k)

    results = []
    for score, idx in zip(values, indices):
        results.append({
            "image_path": IMAGE_PATHS[idx.item()],
            "similarity": f"{score.item() * 100:.2f}%"
        })
    return results


app = FastAPI(title="Image Retrieval Search Engine API")

@app.post("/search")
async def search_engine(
    model_type: str = Form(...),      # "convnext", "dinov2", "combined", "siglip"
    query_type: str = Form("image"),
    text_query: str = Form(""),
    top_k: int = Form(5),
    file: UploadFile = File(None)
):
    try:
        model_type = model_type.lower()
        query_type = query_type.lower()

        with torch.no_grad():
            # IMAGE QUERY
            if query_type == "image":
                if not file: return {"error": "Thiếu file ảnh tải lên!"}
                image_bytes = await file.read()
                image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

                # Tính đặc trưng tùy theo mô hình
                if model_type == "convnext":
                    inputs = transform_convnext(image).unsqueeze(0).to(device)
                    q_vec = convnext_model(inputs)
                    q_vec = q_vec / q_vec.norm(p=2, dim=-1, keepdim=True)
                    results = retrieve_top_k(q_vec, DB_CONVNEXT, top_k)

                elif model_type == "dinov2":
                    inputs = dino_processor(images=image, return_tensors="pt").to(device)
                    outputs = dino_model(**inputs)
                    q_vec = outputs.last_hidden_state[:, 0, :]
                    q_vec = q_vec / q_vec.norm(p=2, dim=-1, keepdim=True)
                    results = retrieve_top_k(q_vec, DB_DINO, top_k)

                elif model_type == "combined":
                    # ConvNeXt
                    in_conv = transform_convnext(image).unsqueeze(0).to(device)
                    vec_conv = convnext_model(in_conv)
                    vec_conv = vec_conv / vec_conv.norm(p=2, dim=-1, keepdim=True)

                    # DINOv2
                    in_dino = dino_processor(images=image, return_tensors="pt").to(device)
                    out_dino = dino_model(**in_dino)
                    vec_dino = out_dino.last_hidden_state[:, 0, :]
                    vec_dino = vec_dino / vec_dino.norm(p=2, dim=-1, keepdim=True)

                    # Concat & Normalize
                    q_vec = torch.cat([vec_conv, vec_dino], dim=-1)
                    q_vec = q_vec / q_vec.norm(p=2, dim=-1, keepdim=True)
                    results = retrieve_top_k(q_vec, DB_COMBINED, top_k)

                elif model_type == "siglip":
                    inputs = siglip_processor(images=image, return_tensors="pt").to(device)
                    q_vec = siglip_model.get_image_features(**inputs)

                    if hasattr(q_vec, 'pooler_output'):
                        q_vec = q_vec.pooler_output

                    q_vec = q_vec / q_vec.norm(p=2, dim=-1, keepdim=True)
                    results = retrieve_top_k(q_vec, DB_SIGLIP, top_k)

                else:
                    return {"error": "model_type không hợp lệ!"}

            # TEXT QUERY
            elif query_type == "text":
                if model_type != "siglip":
                    return {"error": f"Mô hình {model_type} không hỗ trợ Text. Dùng siglip!"}
                if not text_query.strip():
                    return {"error": "Bạn chưa nhập chữ!"}

                # SigLIP Processor xử lý Text rất mượt
                inputs = siglip_processor(text=[text_query], padding="max_length", return_tensors="pt").to(device)
                q_vec = siglip_model.get_text_features(**inputs)

                if hasattr(q_vec, 'pooler_output'):
                        q_vec = q_vec.pooler_output

                q_vec = q_vec / q_vec.norm(p=2, dim=-1, keepdim=True)
                results = retrieve_top_k(q_vec, DB_SIGLIP, top_k)

            else:
                return {"error": "query_type chỉ hỗ trợ 'image' hoặc 'text'"}

        return {
            "query_type": query_type,
            "model_used": model_type,
            "top_k_results": results
        }

    except Exception as e:
        return {"error": str(e)}


os.system("fuser -k 8000/tcp")

from google.colab import userdata

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

public_url = ngrok.connect(8000).public_url
print(f"🌟 SEARCH ENGINE ĐÃ ONLINE TẠI: {public_url}")
print(f"   (Test giao diện trực tiếp tại: {public_url}/docs)")

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

🌟 SEARCH ENGINE ĐÃ ONLINE TẠI: https://prince-glebal-dusty.ngrok-free.dev
   (Test giao diện trực tiếp tại: https://prince-glebal-dusty.ngrok-free.dev/docs)


INFO:     Started server process [1503]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2402:9d80:85d:2740:cd4f:fd50:4385:ebf:0 - "POST /search HTTP/1.1" 200 OK
INFO:     2402:9d80:85d:2740:cd4f:fd50:4385:ebf:0 - "POST /search HTTP/1.1" 200 OK
INFO:     2402:9d80:85d:2740:cd4f:fd50:4385:ebf:0 - "POST /search HTTP/1.1" 200 OK
INFO:     2402:9d80:85d:2740:cd4f:fd50:4385:ebf:0 - "POST /search HTTP/1.1" 200 OK
INFO:     2402:9d80:85d:2740:cd4f:fd50:4385:ebf:0 - "POST /search HTTP/1.1" 200 OK
INFO:     2402:9d80:85d:2740:cd4f:fd50:4385:ebf:0 - "POST /search HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1503]
